# Registro de Errores · Silver Pedidos

## Propósito

Registrar en `etl_log` el error de la task `slv_orders` y enviar una alerta a Slack. Se ejecuta únicamente cuando esa task falla.

## Flujo

1. Recuperar el `log_id` publicado por `slv_orders` y recibir el código de error, el `run_id` y el resultado desde el Workflow.
2. Consultar el detalle de la excepción mediante la Databricks Jobs API.
3. Actualizar el registro en `etl_log` con estado `ERROR`, la fecha de finalización y el detalle técnico.
4. Enviar la alerta a Slack con la task, el estado, el código de error, el `run_id` y el `log_id`.

## Tabla de trazabilidad

`metadata_dataco.supply_chain.hd_etl_log`

## Dependencias

Depende únicamente de `slv_orders` y se ejecuta con la condición **At least one failed**. Ninguna otra task del Workflow depende de este notebook.

## Resultado esperado

El registro de la ejecución fallida actualizado con estado `ERROR` y su detalle técnico, y la alerta correspondiente enviada a Slack.

In [0]:
# ============================================================
# OBTENCIÓN DEL LOG_ID Y PARÁMETROS DEL TASK FALLIDO
# ============================================================

dbutils.widgets.text("error_code", "UNKNOWN_ERROR")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("result_state", "FAILED")

error_code = dbutils.widgets.get("error_code")
run_id = dbutils.widgets.get("run_id")
result_state = dbutils.widgets.get("result_state")

log_id = dbutils.jobs.taskValues.get(
    taskKey="slv_orders",
    key="log_id",
    debugValue=""
)

print(f"✓ log_id: {log_id}")
print(f"✓ error_code: {error_code}")
print(f"✓ run_id: {run_id}")
print(f"✓ result_state: {result_state}")

In [0]:
# ============================================================
# OBTENCIÓN DEL DETALLE DEL ERROR MEDIANTE JOBS API
# ============================================================

import requests

error_detail = None

if run_id and str(run_id).isdigit() and str(run_id) != "0":
    try:
        workspace_url = spark.conf.get("spark.databricks.workspaceUrl")

        token = (
            dbutils.notebook.entry_point
            .getDbutils()
            .notebook()
            .getContext()
            .apiToken()
            .get()
        )

        url = f"https://{workspace_url}/api/2.2/jobs/runs/get-output"

        response = requests.get(
            url,
            headers={"Authorization": f"Bearer {token}"},
            params={"run_id": str(run_id)}
        )

        if response.status_code == 200:
            output_info = response.json()
            error_detail = (
                output_info.get("error")
                or output_info.get("error_trace")
                or (output_info.get("notebook_output") or {}).get("result")
            )
        else:
            error_detail = f"Error al consultar la API (Status {response.status_code}): {response.text}"

    except Exception as ex:
        error_detail = f"Excepción durante la conexión a Jobs API: {str(ex)}"

if not error_detail:
    error_detail = "No fue posible obtener la traza del error mediante Jobs API (Run ID no válido o sin respuesta)."

print("✓ Detalle del error obtenido:")
print(error_detail)

In [0]:
# ============================================================
# CONSTRUCCIÓN DEL MENSAJE DE ERROR
# ============================================================

mensaje_error = (
    f"Task: slv_orders | "
    f"Error code: {error_code} | "
    f"Run ID: {run_id} | "
    f"Resultado: {result_state} | "
    f"Detalle: {error_detail}"
)

print("Mensaje de error:")
print(mensaje_error)

In [0]:
# ============================================================
# ACTUALIZACIÓN DEL LOG COMO ERROR
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql import functions as F

delta_log = DeltaTable.forName(spark, "metadata_dataco.supply_chain.hd_etl_log")

if log_id:
    delta_log.update(
        condition=F.col("log_id") == log_id,
        set={
            "fecha_fin": F.current_timestamp(),
            "estado": F.lit("ERROR"),
            "mensaje_error": F.lit(mensaje_error[:2000])
        }
    )
    print(f"✓ etl_log actualizado como ERROR: {log_id}")
else:
    print("⚠️ No se realizó la actualización: log_id no fue recibido.")

In [0]:
# ============================================================
# ALERTA A SLACK
# ============================================================

import requests as slack_requests

try:
    webhook_url = dbutils.secrets.get(scope="alerts", key="slack_webhook_url")

    slack_payload = {
        "text": (
            f":rotating_light: *Fallo en pipeline DataCo*\n"
            f"*Task:* slv_orders\n"
            f"*Estado:* {result_state}\n"
            f"*Error code:* {error_code}\n"
            f"*Run ID:* {run_id}\n"
            f"*log_id:* {log_id}"
        )
    }

    slack_response = slack_requests.post(webhook_url, json=slack_payload, timeout=10)

    if slack_response.status_code == 200:
        print("✓ Alerta enviada a Slack")
    else:
        print(f"⚠️ Slack respondió con status {slack_response.status_code}")

except Exception as ex:
    print(f"⚠️ No se pudo enviar la alerta a Slack (scope/clave no configurado aún): {ex}")